<a href="https://colab.research.google.com/github/natthoncstu/SP_Project/blob/main/CS403_Report_65_1_13_tpb_r1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

เชื่อมต่อ drive


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


รับ dataset


In [ ]:
import pandas as pd
import numpy as np

dent_reviewers = pd.read_csv("/content/drive/MyDrive/SP_project_matching/dent_reviewers.csv")
dent_papers = pd.read_csv("/content/drive/MyDrive/SP_project_matching/dent_papers.csv")

art_reviewers = pd.read_csv("/content/drive/MyDrive/SP_project_matching/art_reviewers.csv")
art_papers = pd.read_csv("/content/drive/MyDrive/SP_project_matching/art_papers.csv")

arch_reviewers = pd.read_csv("/content/drive/MyDrive/SP_project_matching/architect_reviewers.csv")
arch_papers = pd.read_csv("/content/drive/MyDrive/SP_project_matching/architect_papers.csv")

all_reviewers = pd.concat([dent_reviewers,art_reviewers,arch_reviewers], ignore_index=True)
all_papers = pd.concat([dent_papers,art_papers,arch_papers], ignore_index=True)

Clean Data

In [ ]:
all_reviewers["text"] = (
    all_reviewers["title"] + " " + all_reviewers["abstract"]
).str.lower()

all_papers["text"] = (
    all_papers["title"] + " " + all_papers["abstract"]
).str.lower()

Tokenize

In [ ]:
pip install pythainlp

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.3/19.3 MB 57.2 MB/s eta 0:00:00


In [ ]:
from pythainlp.tokenize import word_tokenize

reviewer_tokens = []
for text in all_reviewers["text"]:
    reviewer_tokens.append(word_tokenize(text, keep_whitespace=False))

paper_tokens = []
for text in all_papers["text"]:
    paper_tokens.append(word_tokenize(text, keep_whitespace=False))

all_reviewers["tokens"] = reviewer_tokens
all_papers["tokens"] = paper_tokens

TF-IDF preparation

In [ ]:
from pythainlp.corpus import thai_stopwords

stop_words = thai_stopwords()

tfidf_reviewer_tokens = []
for tokens in reviewer_tokens:
    filtered = [w for w in tokens if w not in stop_words]
    tfidf_reviewer_tokens.append(filtered)

tfidf_paper_tokens = []
for tokens in paper_tokens:
    filtered = [w for w in tokens if w not in stop_words]
    tfidf_paper_tokens.append(filtered)

all_reviewers["tfidf_tokens"] = tfidf_reviewer_tokens
all_papers["tfidf_tokens"] = tfidf_paper_tokens

all_tfidf_tokens = tfidf_paper_tokens + tfidf_reviewer_tokens
all_texts = [" ".join(tokens) for tokens in all_tfidf_tokens]

TF-IDF Vectorization

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
vectorizer = TfidfVectorizer()
vectorizer.fit(all_texts)

reviewer_texts = [" ".join(tokens) for tokens in tfidf_reviewer_tokens]
paper_texts = [" ".join(tokens) for tokens in tfidf_paper_tokens]

vec_tfidf_reviewer = vectorizer.transform(reviewer_texts)
vec_tfidf_paper = vectorizer.transform(paper_texts)

Doc2Vec


In [ ]:
pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 37.2 MB/s eta 0:00:00


In [ ]:
from gensim.models.doc2vec import Doc2Vec, TaggedDocument

documents = []

for i, tokens in enumerate(reviewer_tokens):
    documents.append(
        TaggedDocument(words=tokens, tags=[f"reviewers_{i}"])
    )

for j, tokens in enumerate(paper_tokens):
    documents.append(
        TaggedDocument(words=tokens, tags=[f"papers_{j}"])
    )

model = Doc2Vec(
    vector_size=300,
    window=5,
    min_count=2,
    workers=4,
    epochs=40,
    dm=1
)

model.build_vocab(documents)
model.train(
    documents,
    total_examples=model.corpus_count,
    epochs=model.epochs
)


In [ ]:
vec_doc2vec_reviewers = [
    model.dv[f"reviewers_{i}"] for i in range(len(reviewer_tokens))
]

vec_doc2vec_papers = [
    model.dv[f"papers_{j}"] for j in range(len(paper_tokens))
]


SBERT embedding


In [ ]:
from sentence_transformers import SentenceTransformer

In [ ]:
model = SentenceTransformer('intfloat/multilingual-e5-base')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/418 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('intfloat/multilingual-e5-base')

vec_sbert_reviewers = model.encode(all_reviewers["text"], convert_to_numpy=True)
vec_sbert_papers = model.encode(all_papers["text"], convert_to_numpy=True)

Cosine Similarity

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

tfidf_sim = cosine_similarity(vec_tfidf_reviewer, vec_tfidf_paper)
doc2vec_sim = cosine_similarity(vec_doc2vec_reviewers, vec_doc2vec_papers)
sbert_sim = cosine_similarity(vec_sbert_reviewers, vec_sbert_papers)

In [ ]:
print("Affinity Score ที่ได้จาก TF-IDF")
print(tfidf_sim.round(3))
print("Affinity Score ที่ได้จาก Doc2Vec")
print(doc2vec_sim.round(3))
print("Affinity Score ที่ได้จาก SBERT")
print(sbert_sim.round(3))

Affinity Score ที่ได้จาก TF-IDF
[[0.032 0.025 0.063 ... 0.02  0.062 0.062]
 [0.019 0.03  0.042 ... 0.066 0.085 0.072]
 [0.022 0.095 0.061 ... 0.012 0.049 0.024]
 ...
 [0.018 0.018 0.015 ... 0.035 0.054 0.076]
 [0.045 0.015 0.053 ... 0.022 0.063 0.037]
 [0.021 0.009 0.013 ... 0.114 0.034 0.037]]
Affinity Score ที่ได้จาก Doc2Vec
[[ 0.401  0.364  0.588 ...  0.269  0.289  0.484]
 [ 0.215  0.199  0.446 ...  0.227  0.443  0.354]
 [ 0.464  0.668  0.444 ...  0.086  0.33   0.112]
 ...
 [-0.056  0.244  0.144 ...  0.312  0.509  0.448]
 [ 0.096  0.305  0.209 ... -0.06   0.496  0.187]
 [ 0.044  0.113  0.234 ...  0.44   0.312  0.145]]
Affinity Score ที่ได้จาก SBERT
[[0.848 0.857 0.893 ... 0.831 0.846 0.839]
 [0.817 0.833 0.885 ... 0.854 0.856 0.857]
 [0.846 0.86  0.883 ... 0.796 0.838 0.815]
 ...
 [0.832 0.838 0.827 ... 0.829 0.864 0.846]
 [0.831 0.844 0.833 ... 0.807 0.847 0.844]
 [0.811 0.822 0.82  ... 0.847 0.849 0.841]]


Hungarian Algorithm

In [ ]:
def hungarian_with_mask_fixed(
    sim,
    max_per_paper
):
    import numpy as np
    from scipy.optimize import linear_sum_assignment

    n_reviewers, n_papers = sim.shape
    reviewer_count = np.zeros(n_reviewers, dtype=int)
    paper_count = np.zeros(n_papers, dtype=int)

    max_per_reviewer = max_per_paper*n_papers/n_reviewers

    results = []
    assigned = set()

    round_num = 0
    while True:
        round_num += 1

        mask_reviewers = reviewer_count < max_per_reviewer
        mask_papers = paper_count < max_per_paper

        if not mask_reviewers.any() or not mask_papers.any(): #check available
            break

        sub_sim = sim[np.ix_(mask_reviewers, mask_papers)]
        cost = 1 - sub_sim

        active_r = np.where(mask_reviewers)[0]
        active_p = np.where(mask_papers)[0]
        r_to_sub = {r_idx: sub_idx for sub_idx, r_idx in enumerate(active_r)}
        p_to_sub = {p_idx: sub_idx for sub_idx, p_idx in enumerate(active_p)}

        for r_idx, p_idx in assigned:
            if r_idx in r_to_sub and p_idx in p_to_sub:
                cost[r_to_sub[r_idx], p_to_sub[p_idx]] = np.inf

        if sub_sim.size == 0 or np.all(np.isinf(cost)):
            break

        row_ind, col_ind = linear_sum_assignment(cost)

        real_r = active_r[row_ind]
        real_p = active_p[col_ind]

        for reviewers, papers in zip(real_r, real_p):
            if (reviewers, papers) in assigned:
                continue

            if reviewer_count[reviewers] < max_per_reviewer and paper_count[papers] < max_per_paper:
                score = sim[reviewers, papers]
                results.append((reviewers, papers, score, round_num))
                reviewer_count[reviewers] += 1
                paper_count[papers] += 1
                assigned.add((reviewers, papers))

    return results

สรุปผล

In [ ]:
dent_reviewers['field'] = 'dent'
dent_papers['field'] = 'dent'
art_reviewers['field'] = 'art'
art_papers['field'] = 'art'
arch_reviewers['field'] = 'arch'
arch_papers['field'] = 'arch'

all_reviewers = pd.concat([dent_reviewers, art_reviewers, arch_reviewers], ignore_index=True)
all_papers = pd.concat([dent_papers, art_papers, arch_papers], ignore_index=True)

TF-IDF

In [ ]:
assignments = hungarian_with_mask_fixed(
    tfidf_sim,
    3
)
reviewer_ids = all_reviewers["id"]
paper_ids = all_papers["id"]

reviewer_group = all_reviewers["field"]
paper_group = all_papers["field"]

records = []
for r, p, score, round_num in assignments:
    records.append({
        "reviewer_id": reviewer_ids[r],
        "paper_id": paper_ids[p],
        "score": score.round(3),
        "reviewer_group": reviewer_group[r],
        "paper_group": paper_group[p],
        "correct_category": reviewer_group[r] == paper_group[p],
        "round": round_num
    })

assignments_tfidf_df = pd.DataFrame(records)

display(assignments_tfidf_df)

,reviewer_id,paper_id,score,reviewer_group,paper_group,correct_category,round
0,dent_reviewer_1,dent_paper_28,0.131,dent,dent,True,1
1,dent_reviewer_2,dent_paper_40,0.396,dent,dent,True,1
2,dent_reviewer_3,dent_paper_35,0.382,dent,dent,True,1
3,dent_reviewer_4,dent_paper_50,0.110,dent,dent,True,1
4,dent_reviewer_5,dent_paper_46,0.268,dent,dent,True,1
...,...,...,...,...,...,...,...
445,arch_reviewer_2,dent_paper_22,0.034,arch,dent,False,16
446,arch_reviewer_10,arch_paper_46,0.033,arch,arch,True,16
447,dent_reviewer_4,dent_paper_1,0.005,dent,dent,True,17
448,art_reviewer_2,arch_paper_46,0.021,art,arch,False,17


In [ ]:
print(f"ASSIGNMENT STATISTICS TF-IDF")

avg_score = assignments_tfidf_df.groupby('reviewer_group')['score'].mean()
correct_matches = assignments_tfidf_df.groupby('reviewer_group')['correct_category'].sum()
total_assignments = assignments_tfidf_df.groupby('reviewer_group')['reviewer_id'].count()

tfidf_stat_df = pd.DataFrame({
    'Avg_Affinity_Score': avg_score.round(3),
    'Correct_Matches': correct_matches,
    'Total_Assignments': total_assignments
})

tfidf_stat_df['Accuracy_%'] = (tfidf_stat_df['Correct_Matches'] / tfidf_stat_df['Total_Assignments'] * 100).round(2)

all_avg_score = assignments_tfidf_df['score'].mean()
all_correct_matches = assignments_tfidf_df['correct_category'].sum()
all_total_assignments = assignments_tfidf_df['reviewer_id'].count()
all_accuracy = (all_correct_matches / all_total_assignments * 100).round(2)

total_doc2vec_stats_df = pd.DataFrame({
    'Avg_Affinity_Score': [all_avg_score.round(3)],
    'Correct_Matches': [all_correct_matches],
    'Total_Assignments': [all_total_assignments],
    'Accuracy_%': [all_accuracy]
}, index=['total'])

tfidf_stat_df = pd.concat([tfidf_stat_df, total_doc2vec_stats_df])
display(tfidf_stat_df)

ASSIGNMENT STATISTICS TF-IDF


,Avg_Affinity_Score,Correct_Matches,Total_Assignments,Accuracy_%
arch,0.133,110,150,73.33
art,0.177,127,150,84.67
dent,0.124,118,150,78.67
total,0.144,355,450,78.89


Doc2Vec

In [ ]:
assignments = hungarian_with_mask_fixed(
    doc2vec_sim,
    3
)
reviewer_ids = all_reviewers["id"]
paper_ids = all_papers["id"]

reviewer_group = all_reviewers["field"]
paper_group = all_papers["field"]

records = []
for r, p, score, round_num in assignments:
    records.append({
        "reviewer_id": reviewer_ids[r],
        "paper_id": paper_ids[p],
        "score": score.round(3),
        "reviewer_group": reviewer_group[r],
        "paper_group": paper_group[p],
        "correct_category": reviewer_group[r] == paper_group[p],
        "round": round_num
    })

assignments_doc2vec_df = pd.DataFrame(records)

display(assignments_doc2vec_df)

,reviewer_id,paper_id,score,reviewer_group,paper_group,correct_category,round
0,dent_reviewer_1,dent_paper_14,0.673,dent,dent,True,1
1,dent_reviewer_2,dent_paper_40,0.838,dent,dent,True,1
2,dent_reviewer_3,dent_paper_35,0.743,dent,dent,True,1
3,dent_reviewer_4,dent_paper_9,0.780,dent,dent,True,1
4,dent_reviewer_5,dent_paper_12,0.898,dent,dent,True,1
...,...,...,...,...,...,...,...
445,art_reviewer_1,arch_paper_37,0.293,art,arch,False,16
446,art_reviewer_7,art_paper_34,0.316,art,art,True,16
447,arch_reviewer_2,arch_paper_1,0.303,arch,arch,True,16
448,arch_reviewer_10,arch_paper_3,0.346,arch,arch,True,16


In [ ]:
print(f"ASSIGNMENT STATISTICS Doc2Vec")

avg_score = assignments_doc2vec_df.groupby('reviewer_group')['score'].mean()
correct_matches = assignments_doc2vec_df.groupby('reviewer_group')['correct_category'].sum()
total_assignments = assignments_doc2vec_df.groupby('reviewer_group')['reviewer_id'].count()

doc2vec_stat_df = pd.DataFrame({
    'Avg_Affinity_Score': avg_score.round(3),
    'Correct_Matches': correct_matches,
    'Total_Assignments': total_assignments
})

doc2vec_stat_df['Accuracy_%'] = (doc2vec_stat_df['Correct_Matches'] / doc2vec_stat_df['Total_Assignments'] * 100).round(2)

all_avg_score = assignments_doc2vec_df['score'].mean()
all_correct_matches = assignments_doc2vec_df['correct_category'].sum()
all_total_assignments = assignments_doc2vec_df['reviewer_id'].count()
all_accuracy = (all_correct_matches / all_total_assignments * 100).round(2)

total_doc2vec_stats_df = pd.DataFrame({
    'Avg_Affinity_Score': [all_avg_score.round(3)],
    'Correct_Matches': [all_correct_matches],
    'Total_Assignments': [all_total_assignments],
    'Accuracy_%': [all_accuracy]
}, index=['total'])

doc2vec_stat_df = pd.concat([doc2vec_stat_df, total_doc2vec_stats_df])
display(doc2vec_stat_df)

ASSIGNMENT STATISTICS Doc2Vec


,Avg_Affinity_Score,Correct_Matches,Total_Assignments,Accuracy_%
arch,0.540,128,150,85.33
art,0.657,140,150,93.33
dent,0.570,131,150,87.33
total,0.589,399,450,88.67


SBERT

In [ ]:
assignments = hungarian_with_mask_fixed(
    sbert_sim,
    3
)
reviewer_ids = all_reviewers["id"]
paper_ids = all_papers["id"]

reviewer_group = all_reviewers["field"]
paper_group = all_papers["field"]

records = []
for r, p, score, round_num in assignments:
    records.append({
        "reviewer_id": reviewer_ids[r],
        "paper_id": paper_ids[p],
        "score": score.round(3),
        "reviewer_group": reviewer_group[r],
        "paper_group": paper_group[p],
        "correct_category": reviewer_group[r] == paper_group[p],
        "round": round_num
    })

assignments_sbert_df = pd.DataFrame(records)

display(assignments_sbert_df)

,reviewer_id,paper_id,score,reviewer_group,paper_group,correct_category,round
0,dent_reviewer_1,dent_paper_3,0.893,dent,dent,True,1
1,dent_reviewer_2,dent_paper_17,0.904,dent,dent,True,1
2,dent_reviewer_3,dent_paper_35,0.904,dent,dent,True,1
3,dent_reviewer_4,dent_paper_50,0.921,dent,dent,True,1
4,dent_reviewer_5,dent_paper_20,0.927,dent,dent,True,1
...,...,...,...,...,...,...,...
445,arch_reviewer_3,art_paper_8,0.819,arch,art,False,16
446,arch_reviewer_6,art_paper_19,0.804,arch,art,False,16
447,arch_reviewer_7,art_paper_12,0.794,arch,art,False,16
448,arch_reviewer_8,art_paper_32,0.812,arch,art,False,16


In [ ]:
print(f"ASSIGNMENT STATISTICS SBERT")

avg_score = assignments_sbert_df.groupby('reviewer_group')['score'].mean()
correct_matches = assignments_sbert_df.groupby('reviewer_group')['correct_category'].sum()
total_assignments = assignments_sbert_df.groupby('reviewer_group')['reviewer_id'].count()

sbert_stat_df = pd.DataFrame({
    'Avg_Affinity_Score': avg_score.round(3),
    'Correct_Matches': correct_matches,
    'Total_Assignments': total_assignments
})

sbert_stat_df['Accuracy_%'] = (sbert_stat_df['Correct_Matches'] / sbert_stat_df['Total_Assignments'] * 100).round(2)

all_avg_score = assignments_sbert_df['score'].mean()
all_correct_matches = assignments_sbert_df['correct_category'].sum()
all_total_assignments = assignments_sbert_df['reviewer_id'].count()
all_accuracy = (all_correct_matches / all_total_assignments * 100).round(2)

total_stats_df = pd.DataFrame({
    'Avg_Affinity_Score': [all_avg_score.round(3)],
    'Correct_Matches': [all_correct_matches],
    'Total_Assignments': [all_total_assignments],
    'Accuracy_%': [all_accuracy]
}, index=['total'])

sbert_stat_df = pd.concat([sbert_stat_df, total_stats_df])
display(sbert_stat_df)


ASSIGNMENT STATISTICS SBERT


,Avg_Affinity_Score,Correct_Matches,Total_Assignments,Accuracy_%
arch,0.858,132,150,88.00
art,0.866,132,150,88.00
dent,0.881,138,150,92.00
total,0.868,402,450,89.33
